In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

In [3]:
#Dynamic Path Detection so it runs on all user environments
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")

In [4]:
#Spark Initialization
spark = SparkSession.builder \
    .appName("BODS_Predictive_Engine") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 10:55:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/02 10:55:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [7]:
# Load files
df_cat = spark.read.csv(os.path.join(DATA_DIR, "overall_data_catalogue.csv"), header=True, inferSchema=True)
df_comp = spark.read.csv(os.path.join(DATA_DIR, "overall_compliance_report.csv"), header=True, inferSchema=True)

# Surgical Join: Joining on Service Code (Catalogue) and Service Number (Compliance)
df_unified = df_cat.join(
    df_comp, 
    df_cat["Service Code"] == df_comp["Registration:Service Number"], 
    how="right"
)

print(f"Row count after surgical join: {df_unified.count()}")

[Stage 24:>                                                         (0 + 2) / 2]

Row count after surgical join: 103437
